# Cincinatti car crashes

## Dataset definition

All reported Cincinnati, Ohio car crashes since 2010.

This dataset includes fatal, injury, and non-injury crashes. In compliance with privacy laws, all Public Safety datasets are anonymized

[Original dataset](https://data.cincinnati-oh.gov/safety/Traffic-Crash-Reports-CPD-/rvmt-pkmq/about_data)

**Data Dictionary:**
![image](Data-Dictionary.png)


### Inspiration
- Find which areas of town are the sources of most crashes?
- Figure out how much weather changes the frequency of crashes.
- Which community has the most crashes?

In [ ]:
# # Instalar dependencias
# !pip install kagglehub
# !pip install rich
# !pip install pandas


In [ ]:
import os
import kagglehub    
from rich import print
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
# configuracion de API de Kaggle
os.environ["KAGGLE_USERNAME"] = "PruebasPython"
os.environ["KAGGLE_KEY"] = "KGAT_1f9b6c227b7aa05264968719b0feed91"


In [ ]:
# Descarga del dataset
path = kagglehub.dataset_download("steverusso/cincinnati-car-crash-data")
print("Path to dataset files:", path)

print("Descargando el dataset")
dataset = pd.read_csv(path+"/cincinnati_traffic_crash_data__cpd.csv") # Carga datos desde un CSV y devuelve un DataFrame 
# dataset.to_csv("archivo.csv", index=False)

dataset.head()

In [ ]:
# Exploracion de datos

print("Info")
print(dataset.info())
print("Valores nulos")
print(dataset.isnull().sum())

## 1. Limpieza inicial de variables

### 1.1  Variables de ID de registros
- La columna ```Unnamed: 0``` corresponde al número de fila, luego la eliminamos porque no aporta información.
- Las columnas ```INSTANCEID``` y ```LOCALREPORTNO``` se refieren al identificador de una incidencia de accidente y al identificador del informe del accidente, respectivamente. No queremos entrenar sobre valores que se refieran a registros.

Se eliminan estas columnas:

In [ ]:
print(f"Tamaño del dataset original: {dataset.shape}")
dataset = dataset.drop(columns=["Unnamed: 0","INSTANCEID", "LOCALREPORTNO"])
print(f"Tamaño del dataset tras eliminar columnas de ID de registros: {dataset.shape}")

### 1.2  Variables de ubicación redundantes

Las siguientes variables sirven para ubicar geográficamente el lugar del incidente. Todas ellas hacen referencia a la ubicación donde ha ocurrido el accidente, pero lo hacen con distintos niveles de precisión. Es conviene reducir redundancia para quedarse con las que aporten más información.

- ```ADDRESS_X```: dirección del accidente, anonimizada a nivel de bloque.
- ```LATITUDE_X```: coordenada geográfica de latitud.
- ```LONGITUDE_X```: coordenada geográfica de longitud.
- ```CPD_NEIGHBORHOOD```: barrio según la división de la policía (Cincinnati Police Department).
- ```SNA_NEIGHBORHOOD```: barrio estadístico de la ciudad de Cincinnati.
- ```ZIP```: código postal.

In [ ]:
geo_columns = ["ADDRESS_X", "CPD_NEIGHBORHOOD", "ZIP", "SNA_NEIGHBORHOOD"]

# Cantidad de valores únicos
for col in geo_columns:
    print(f"{col}: {dataset[col].nunique()} valores únicos")

# Figura con 2 filas y 2 columnas
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()  # para iterar fácilmente

for i, col in enumerate(geo_columns):
    # Top 20 valores más frecuentes
    top_values = dataset[col].value_counts().head(20)
    
    axes[i].bar(top_values.index.astype(str), top_values.values)
    axes[i].set_title(f"Top 20 valores de {col}", fontsize=14)
    axes[i].set_xlabel(col, fontsize=12)
    axes[i].set_ylabel("Frecuencia", fontsize=12)
    axes[i].tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.show()

En primer lugar, se debe descartar la columna ```ADDRESS_X``` ya que sus valores han sido modificados y toma demasiados valores categóricos únicos para que merezca la pena hacer un _encoding_ sobre ellos.

A continuación, parece razonable mantener las coordenadas geográficas ya que son valores de tipo _float_, que podrán ser usados con facilidad en el proceso de entrenamiento.

Finalmente, las variables ```CPD_NEIGHBORHOOD```, ```SNA_NEIGHBORHOOD``` y ```ZIP``` representan agrupaciones dadas ciertas coordenadas geográficas. Puesto que todas ellas representan un tipo de información similar, tomamos inicialmente ```SNA_NEIGHBORHOOD``` para los análisis, aunque podría ser interesante repetir los resultados obtenidos para las otras. Se ha escogido esta principalmente porque la ciudad de Cincinatti ya había hehco una agrupación estadística de los barrios, además de que tiene pocas categorías únicas.

In [ ]:
dataset = dataset.drop(columns=["ADDRESS_X","CPD_NEIGHBORHOOD", "ZIP"])
print(f"Tamaño del dataset tras eliminar columnas de variables de ubicación redundantes: {dataset.shape}")

### 1.3  Variables categóricas idénticas

Se ha observado que las variables ```CRASHSEVERITYID``` y ```CRASHSEVERITY``` representan la misma información, ya que existe una correspondencia uno a uno entre ambas. Se decide eliminar la de ```CRASHSEVERITYID``` porque se empleará la otra para el mapa de valores de severidad que se creará más adelante. Realmente es indiferente cuál de las dos eliminar, pero se conserva ```CRASHSEVERITY``` por ser más clara para esta priemra fase.

In [ ]:
# identificar relación uno a uno de los valores
print(dataset.groupby("CRASHSEVERITY")["CRASHSEVERITYID"].unique())
print(dataset.groupby("CRASHSEVERITY")["CRASHSEVERITYID"].nunique())

dataset = dataset.drop(columns=["CRASHSEVERITYID"])
print(f"Tamaño del dataset tras eliminar la columna CRASHSEVERITYID: {dataset.shape}")

Tras esta limpieza inicial hemos pasado de 27 variables (columnas) a 20, que, en prinicipio, cuentan con información útil para entrenar los modelos sobre sus instancias.

In [ ]:
dataset = dataset.drop_duplicates()
print(f"Tamaño del dataset tras eliminar filas duplicadas: {dataset.shape}")

print("Info actualizada:")
print(dataset.info())
print("Valores nulos actualizado:")
print(dataset.isnull().sum())

## 2) Limpieza y transformación de los valores nulos

## 3) Transformación de los datos

In [ ]:
# TODO transformar fechas a datetime
dataset["CRASHDATE"] = pd.to_datetime(dataset["CRASHDATE"], format="%m/%d/%Y %I:%M:%S %p")
dataset["DATECRASHREPORTED"] = pd.to_datetime(dataset["DATECRASHREPORTED"], format="%m/%d/%Y %I:%M:%S %p")

# TODO transformar variables categoricas a números